<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/816_CBOv2_IntegrationTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a very solid integration test suite — it’s doing the right thing: **invoke the full graph end-to-end and assert executive outputs**, while not breaking CI when seed data isn’t present.

A few tweaks will make it more robust and more “template-grade”.

---

## What’s great (keep)

### ✅ Skip strategy is correct for a template repo

Using `skipif` when `customers.json` isn’t present is exactly what you want when the repo can be copied without data.

### ✅ Assertions are executive-output focused

You’re not overfitting to implementation details. Checking for key sections + report existence is perfect.

### ✅ You included `processing_time`

That’s an important ops invariant to lock in at integration level.

---

## Biggest improvement: don’t hardcode the path relative to the test file

Right now you set:

```python
ROOT = Path(__file__).resolve().parent
CBOv2_DATA_DIR = ROOT / "agents" / "data" / "CBOv2"
```

This only works if:

* the tests live at repo root **and**
* `agents/` is a sibling of the test file

If tests ever move into `tests/`, or someone runs from a different working directory, this breaks.

✅ Better: resolve repo root the same way your loader does (or in a consistent, test-friendly way).

For example:

```python
PROJECT_ROOT = Path(__file__).resolve().parents[1]  # if tests/ folder
# or parents[0] if tests are at root
CBOv2_DATA_DIR = PROJECT_ROOT / "agents" / "data" / "CBOv2"
```

If you want it bulletproof across both layouts, you can implement a tiny helper that searches upward for `agents/`.

---

## Reduce repetition (minor but nice)

You repeat:

```python
config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR))
graph = create_cbo_v2_graph(config)
initial = build_initial_state({"run_id": "integration_test"})
result = graph.invoke(initial)
```

✅ Use a fixture:

```python
@pytest.fixture
def run_result():
    config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR))
    graph = create_cbo_v2_graph(config)
    initial = build_initial_state({"run_id": "integration_test"})
    return graph.invoke(initial)
```

Then each test just uses `run_result`. Makes failures easier to maintain.

---

## Strengthen one key invariant: report should include run metadata fields

Right now you assert `"Run Metadata"` exists, but you can go one level deeper without becoming brittle:

✅ Add checks for:

* `Run ID`
* `Timestamp`
* `Config version`

These are the “auditability contract” fields.

Example:

```python
assert "**Run ID**" in content
assert "**Timestamp**" in content
assert "**Config version**" in content
```

---

## Processing time: clarify what you mean by it

Your last test checks:

```python
assert result["processing_time"] >= 0
```

Good, but to avoid regressions where it becomes `0.0` due to a bug:

✅ Add a weak expectation:

* `<= 60` seconds (or whatever) is too environment-specific.
* Better: just assert it’s a float.

```python
assert isinstance(result["processing_time"], (int, float))
```

…and leave the value range alone.

---

## A subtle improvement: make the tests independent of the output file system location

You check:

```python
Path(result["report_file_path"]).exists()
```

Good.

But if someone runs on a system where `reports_dir` is misconfigured, you’ll fail even though the report generation function worked logically.

✅ Consider overriding reports_dir to a tmp path *in integration tests*, so you know the report is writable.

Example:

```python
config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR), reports_dir=str(tmp_path / "reports"))
```

That makes the integration test deterministic and avoids polluting repo output.

(You’re already doing this in node tests — same idea.)

---

## Summary of recommended changes (highest value)

1. **Fix root path resolution** (most important)
2. Add a **fixture** to avoid repeating the invoke boilerplate
3. Add a few **auditability field asserts** in the report content
4. Optionally **override reports_dir with tmp_path** to keep tests clean and deterministic



In [ ]:
"""
Integration tests for CBOv2 orchestrator.

Full graph invoke; assert no errors, report path set, key sections in report.
Skip when agents/data/CBOv2 is missing so CI without data does not fail.
Run from project root: pytest test_CBOv2_integration.py -v --tb=short
"""
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pytest
from config import CBOv2Config
from agents.CBOv2.orchestrator.nodes import build_initial_state, create_cbo_v2_graph


# Skip integration tests when CBOv2 data is not present (e.g. CI without data)
CBOv2_DATA_DIR = ROOT / "agents" / "data" / "CBOv2"
CUSTOMERS_JSON = CBOv2_DATA_DIR / "customers.json"
requires_cbo_data = pytest.mark.skipif(
    not CUSTOMERS_JSON.exists(),
    reason="CBOv2 data not present (agents/data/CBOv2/customers.json); run with data to execute",
)


@requires_cbo_data
def test_full_graph_invoke_no_errors():
    config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR))
    graph = create_cbo_v2_graph(config)
    initial = build_initial_state({"run_id": "integration_test"})
    result = graph.invoke(initial)
    assert "errors" in result
    assert result["errors"] == [], f"Expected no errors, got: {result['errors']}"


@requires_cbo_data
def test_full_graph_sets_report_path():
    config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR))
    graph = create_cbo_v2_graph(config)
    initial = build_initial_state({"run_id": "integration_test"})
    result = graph.invoke(initial)
    if result.get("errors"):
        pytest.skip(f"Graph had errors: {result['errors']}")
    assert result.get("report_file_path") is not None
    assert Path(result["report_file_path"]).exists()


@requires_cbo_data
def test_full_graph_report_contains_key_sections():
    config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR))
    graph = create_cbo_v2_graph(config)
    initial = build_initial_state({"run_id": "integration_test"})
    result = graph.invoke(initial)
    if result.get("errors"):
        pytest.skip(f"Graph had errors: {result['errors']}")
    path = result.get("report_file_path")
    if not path or not Path(path).exists():
        pytest.skip("No report file")
    content = Path(path).read_text(encoding="utf-8")
    assert "# Cross-Business Customer Intelligence Orchestrator (CBOv2)" in content
    assert "Verdict:" in content or "Verdict" in content
    assert "Executive Summary" in content
    assert "Portfolio Rollup" in content
    assert "Run Metadata" in content
    assert "Top Cross-Business" in content
    assert "No probabilistic scoring" in content or "rule-based" in content.lower()


@requires_cbo_data
def test_full_graph_sets_processing_time():
    config = CBOv2Config(data_dir=str(CBOv2_DATA_DIR))
    graph = create_cbo_v2_graph(config)
    initial = build_initial_state({"run_id": "integration_test"})
    result = graph.invoke(initial)
    if result.get("errors"):
        pytest.skip(f"Graph had errors: {result['errors']}")
    assert "processing_time" in result
    assert result["processing_time"] is not None
    assert result["processing_time"] >= 0
